In [1]:
import os
import json
import uuid
from pathlib import Path

import chromadb
from sentence_transformers import SentenceTransformer
from groq import Groq
from dotenv import load_dotenv

from tqdm import tqdm

c:\Users\lenovo\OneDrive\Desktop\web scrapping\.venv\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [2]:
load_dotenv()

GROQ_API_KEY = os.getenv("GROQ_API_KEY")

print(GROQ_API_KEY[:15] + "...")

gsk_Yr6ak8ydnMP...


In [3]:
client = Groq(
    api_key=GROQ_API_KEY
)

print("Groq Connected Successfully")

Groq Connected Successfully


In [4]:
faculty_folder = Path("faculty")

json_files = sorted(faculty_folder.glob("*.json"))
txt_files = sorted(faculty_folder.glob("*.txt"))

print(f"JSON Files : {len(json_files)}")
print(f"TXT Files  : {len(txt_files)}")

JSON Files : 7
TXT Files  : 7


In [5]:
# Department name mapping from filename
DEPT_MAP = {
    "Artificial_Intelligence": "ai",
    "Computer_Science": "cs",
    "Software_Engineering": "se",
    "Electrical_Engineering": "ee",
    "Cyber_Security": "cyber",
    "Management_Sciences": "ms",
    "Sciences_Humanities": "sh",
    "cyber": "cyber",  # handles cyber_faculty_file.json
}

def get_dept(filename):
    """Extract department key from filename like 'Software_Engineering_faculty_file.json'"""
    for key, val in DEPT_MAP.items():
        if key.lower() in filename.lower():
            return val
    return "unknown"

documents = []

for file in json_files:

    with open(file, "r", encoding="utf-8") as f:
        data = json.load(f)

    dept = get_dept(file.name)

    # Handle both list and dictionary JSON files
    if isinstance(data, list):
        for faculty in data:
            text = ""
            for key, value in faculty.items():
                text += f"{key}: {value}\n"
            documents.append({
                "source": file.name,
                "dept": dept,
                "text": text.strip()
            })

    elif isinstance(data, dict):
        text = ""
        for key, value in data.items():
            text += f"{key}: {value}\n"
        documents.append({
            "source": file.name,
            "dept": dept,
            "text": text.strip()
        })

print("Total Faculty Profiles:", len(documents))


Total Faculty Profiles: 186


In [6]:
print(documents[0]["source"])
print("-" * 60)
print(documents[0]["text"][:1000])

Artificial_Intelligence_faculty_file.json
------------------------------------------------------------
name: Dr. Muhammad Rafi, PhD
designation: Professor and  HOD
email: muhammad.rafi@nu.edu.pk
extension: 222
profile: https://khi.nu.edu.pk/personnel/dr-muhammad-rafi-phd/


In [7]:
texts = []
metadatas = []
ids = []

for i, doc in enumerate(documents):
    texts.append(doc["text"])
    metadatas.append({
        "source": doc["source"],
        "dept": doc["dept"]
    })
    ids.append(f"faculty_{i}")

print("Total Embeddings:", len(texts))


Total Embeddings: 186


In [8]:
embedded_documents = []

for i, doc in enumerate(documents):
    embedded_documents.append({
        "id": f"faculty_{i}",
        "source": doc["source"],
        "dept": doc["dept"],
        "text": doc["text"]
    })

print("Total Faculty Documents:", len(embedded_documents))


Total Faculty Documents: 186


In [9]:
print(embedded_documents[0]["source"])
print("-" * 50)
print(embedded_documents[0]["text"])

Artificial_Intelligence_faculty_file.json
--------------------------------------------------
name: Dr. Muhammad Rafi, PhD
designation: Professor and  HOD
email: muhammad.rafi@nu.edu.pk
extension: 222
profile: https://khi.nu.edu.pk/personnel/dr-muhammad-rafi-phd/


In [10]:
from sentence_transformers import SentenceTransformer

embedding_model = SentenceTransformer(
    "all-MiniLM-L6-v2"
)

print("Embedding model loaded!")

Loading weights: 100%|██████████| 103/103 [00:00<00:00, 4116.84it/s]


Embedding model loaded!


In [11]:
embeddings = embedding_model.encode(
    texts,
    show_progress_bar=True
)

Batches: 100%|██████████| 6/6 [00:04<00:00,  1.38it/s]


In [12]:
chroma_client = chromadb.PersistentClient(
    path="chroma_db"
)

In [13]:
collection = chroma_client.get_or_create_collection(
    name="faculty_rag"
)

print("Collection Ready")

Collection Ready


In [14]:
# Delete the old collection completely
try:
    chroma_client.delete_collection("faculty_rag")
    print("Old collection deleted.")
except:
    print("No previous collection found.")

# Create a fresh collection
collection = chroma_client.get_or_create_collection(
    name="faculty_rag"
)

# Store all faculty profiles — now with dept metadata
for doc in tqdm(embedded_documents):
    embedding = embedding_model.encode(doc["text"]).tolist()
    collection.add(
        ids=[doc["id"]],
        embeddings=[embedding],
        documents=[doc["text"]],
        metadatas=[{
            "source": doc["source"],
            "dept": doc["dept"]
        }]
    )

print("Finished storing embeddings.")
print("Total vectors:", collection.count())


Old collection deleted.


100%|██████████| 186/186 [00:11<00:00, 16.05it/s]

Finished storing embeddings.
Total vectors: 186


In [15]:
print(collection.count())

186


In [16]:
# Department keyword → ChromaDB dept value
QUERY_DEPT_MAP = {
    "software engineering": "se",
    "se dept": "se",
    "electrical engineering": "ee",
    "ee dept": "ee",
    "computer science": "cs",
    "cs dept": "cs",
    "artificial intelligence": "ai",
    "ai dept": "ai",
    "cyber security": "cyber",
    "cybersecurity": "cyber",
    "cyber dept": "cyber",
    "management sciences": "ms",
    "ms dept": "ms",
    "sciences and humanities": "sh",
    "humanities": "sh",
    "sh dept": "sh",
}

def detect_dept(query):
    """Return dept code if a department is mentioned in the query, else None."""
    q = query.lower()
    for phrase, dept in QUERY_DEPT_MAP.items():
        if phrase in q:
            return dept
    return None


def retrieve(query, top_k=5):
    """
    Semantic search with optional department filtering.
    - If the query mentions a specific dept, filter to that dept first.
    - top_k is lowered to 5 (was 10) to keep context tight.
    """
    query_embedding = embedding_model.encode(query).tolist()

    dept = detect_dept(query)

    if dept:
        # Filter to matching department only
        results = collection.query(
            query_embeddings=[query_embedding],
            n_results=top_k,
            where={"dept": {"$eq": dept}}
        )
    else:
        # No dept detected — search all
        results = collection.query(
            query_embeddings=[query_embedding],
            n_results=top_k
        )

    return results


In [17]:
results = retrieve(
    "all the phd holders?"
)

In [18]:
for i in range(len(results["documents"][0])):

    print("=" * 80)

    print("Source :", results["metadatas"][0][i]["source"])

    print()

    print(results["documents"][0][i][:600])

    print()

Source : Computer_Science_faculty_file.json

name: Dr. Nasir Uddin , PhD
designation: Assistant Professor
email: nasir.uddin@nu.edu.pk
extension: 164
profile: https://khi.nu.edu.pk/personnel/dr-nasir-uddin-phd/

Source : Artificial_Intelligence_faculty_file.json

name: Dr. Muhammad Farrukh Shahid , PhD
designation: Assistant Professor
email: mfarrukh.shahid@nu.edu.pk
extension: 163
profile: https://khi.nu.edu.pk/personnel/dr-muhammad-farrukh-shahid-phd/

Source : Computer_Science_faculty_file.json

name: Dr. Nouman Durrani, PhD
designation: Associate Professor
email: muhammad.nouman@nu.edu.pk
extension: 133
profile: https://khi.nu.edu.pk/personnel/dr-nouman-durrani-phd/

Source : Artificial_Intelligence_faculty_file.json

name: Dr. Muhammad Rafi, PhD
designation: Professor and  HOD
email: muhammad.rafi@nu.edu.pk
extension: 222
profile: https://khi.nu.edu.pk/personnel/dr-muhammad-rafi-phd/

Source : Management_Sciences_faculty_file.json

name: Dr. Muhammad Saad, PhD (ON LEAVE)
designati

In [19]:
def build_context(results):
    """
    Combine retrieved chunks into one context string.
    """
    context = ""

    for i, doc in enumerate(results["documents"][0]):
        source = results["metadatas"][0][i]["source"]

        context += f"\n\nSource: {source}\n"
        context += doc

    return context

In [20]:
def create_prompt(question, context):

    return f"""
You are a university faculty information assistant.

Use ONLY the information present in the retrieved context below.
Do NOT use any outside knowledge.

Instructions:
- Read every faculty profile in the context carefully.
- Answer only based on designations/roles mentioned in the profiles.
- If the question asks about a HOD/HoD, look specifically for designation fields
  containing 'HOD', 'HoD', or 'Head of Department' in the RELEVANT department.
- Do NOT confuse HODs from different departments.
- If multiple faculty satisfy the question, list them all.
- If the answer is not in the context, say exactly:
  'I don't have enough information to answer that.'
- Never guess. Never invent names.

Context:
{context}

Question:
{question}

Answer:
"""


In [21]:
def ask_groq(prompt):

    response = client.chat.completions.create(

        model="llama-3.3-70b-versatile",

        messages=[
            {
                "role": "user",
                "content": prompt
            }
        ],

        temperature=0.2
    )

    return response.choices[0].message.content

In [22]:
results = retrieve("Who is the HOD of Cyber Security?")

for i in range(len(results["documents"][0])):
    print("=" * 80)
    print(results["metadatas"][0][i]["source"])
    print()
    print(results["documents"][0][i])

Cyber_Security_faculty_file.json

name: Mr. Muhammad Nouman Rajput
designation: Lecturer
email: nouman.rajput@nu.edu.pk
extension: 287
profile: https://khi.nu.edu.pk/personnel/mr-muhammad-nouman-rajput/
Cyber_Security_faculty_file.json

name: Mr. Muhammad Usman
designation: Lecturer
email: muhammadusman@nu.edu.pk
extension: -
profile: https://khi.nu.edu.pk/personnel/mr-muhammad-usman/
Cyber_Security_faculty_file.json

name: Mr. Sandesh Kumar
designation: Lecturer
email: sandesh.kumar@nu.edu.pk
extension: 172
profile: https://khi.nu.edu.pk/personnel/mr-sandesh-kumar/
Cyber_Security_faculty_file.json

name: Dr. Shahbaz Siddiqui, PhD
designation: Associate Professor, HOD
email: shahbaz.siddiqui@nu.edu.pk
extension: 132
profile: https://khi.nu.edu.pk/personnel/drshahbaz-siddiqui/
Cyber_Security_faculty_file.json

name: Dr. Sufian Hameed , PhD
designation: Professor
email: sufian.hameed@nu.edu.pk
extension: 297
profile: https://khi.nu.edu.pk/personnel/dr-sufian-hameed-phd/


In [25]:
print("=" * 70)
print(" Faculty RAG Chatbot")
print("Type 'exit' or 'quit' to end the chat.")
print("=" * 70)

while True:

    question = input("\nYou: ").strip()

    if question.lower() in ["exit", "quit"]:
        print("\nGoodbye!")
        break

    try:
        # Retrieve relevant chunks
        results = retrieve(question)

        # Build context
        context = build_context(results)

        # Create prompt
        prompt = create_prompt(question, context)

        # Get response from Groq
        answer = ask_groq(prompt)

        # Print answer
        print("\nAssistant:\n")
        print(answer)

        # Print sources
        print("\nSources:")
        sources = sorted(set(m["source"] for m in results["metadatas"][0]))
        for source in sources:
            print(f"  • {source}")

    except Exception as e:
        print(f"\nError: {e}")

    print("\n" + "=" * 70)

 Faculty RAG Chatbot
Type 'exit' or 'quit' to end the chat.



You:  hod of cs dept?



Assistant:

Dr. Fahad Samad

Sources:
  • Computer_Science_faculty_file.json




You:  email of sir fahad samad?



Assistant:

fahad.samad@nu.edu.pk

Sources:
  • Computer_Science_faculty_file.json
  • Sciences_Humanities_faculty_file.json



...


Assistant:

All the faculty members mentioned are from the Artificial Intelligence department, so they all teach Artificial Intelligence. The list includes:

1. Ms. Ramsha Jatt
2. Ms. Alishba Subhani (ON LEAVE)
3. Ms. Sania Urooj
4. Mr. Jahanzaib
5. Dr. Muhammad Yaseen Khan, PhD

Sources:
  • Artificial_Intelligence_faculty_file.json




You:  sir nadeem khan email please and also his desigation?



Assistant:

Mr. Nadeem Khan's email is nadeem.arif@nu.edu.pk and his designation is Lecturer.

Sources:
  • Computer_Science_faculty_file.json
  • Sciences_Humanities_faculty_file.json




You:  exit



Goodbye!
